# Kapitel 8 · Mit pandas transformieren, explorieren und validieren

Dieses Notebook enthält die Codeblöcke des Kapitels in Buchreihenfolge. Die erste Codezelle lädt die Tabellen so, wie sie am Ende des Kapitels „Python-Brücke und kontrollierter Tabellenimport“ vorliegen. Öffnen Sie das Notebook in diesem Ordner und führen Sie die Zellen von oben nach unten aus.

## Ausgangslage: die importierten Tabellen

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
pd.options.mode.copy_on_write = True
inventory_snapshots = pd.read_csv(
    "../daten/inventory_snapshots.csv",
    dtype={"warehouse_id":"Int64", "product_id":"Int64",
           "snapshot_date":"string", "quantity":"Int64"},
    keep_default_na=False, na_values=[""])
product_price_revisions = pd.read_csv(
    "../daten/product_price_revisions.csv",
    dtype={"product_id":"Int64", "price_date":"string", "currency":"string",
           "revision_id":"Int64", "available_at":"string", "price":"Float64"},
    keep_default_na=False, na_values=[""])
orders = pd.read_csv("../kapitel-07-python-import/orders.csv",
    dtype={"order_id":"Int64", "revision_id":"Int64", "customer_id":"Int64",
           "warehouse_id":"Int64", "product_id":"Int64", "currency":"string",
           "event_at":"string", "available_at":"string",
           "order_amount_eur":"Float64", "received_at":"string",
           "processed_at":"string", "processing_group":"string"},
    keep_default_na=False, na_values=[""])
print(inventory_snapshots.to_string(index=False))
print(product_price_revisions.to_string(index=False))


## Datentypen und Konvertierungsfehler sichtbar behandeln

In [ ]:
diagnose = pd.DataFrame({
    "product_id": pd.Series([201, 202, 201], dtype="Int64"),
    "price_text": pd.Series(["100.00", "200,00", pd.NA], dtype="string")})
diagnose["price_numeric"] = pd.to_numeric(
    diagnose["price_text"], errors="coerce").astype("Float64")
diagnose["conversion_error"] = (
    diagnose["price_text"].notna() & diagnose["price_numeric"].isna())
diagnose["source_missing"] = diagnose["price_text"].isna()
print(diagnose.to_string(index=False))


## Datum, Kenntniszeit und Zeitzone konvertieren

In [ ]:
inventory = inventory_snapshots.copy()
prices = product_price_revisions.copy()
inventory["snapshot_date"] = pd.to_datetime(
    inventory["snapshot_date"], format="%Y-%m-%d", errors="raise")
prices["price_date"] = pd.to_datetime(
    prices["price_date"], format="%Y-%m-%d", errors="raise")
prices["available_at"] = pd.to_datetime(
    prices["available_at"], format="ISO8601", utc=True, errors="raise")
cutoff = pd.Timestamp("2026-09-15T06:00:00Z")
print(prices[["product_id", "price_date", "available_at"]].to_string(index=False))
print(prices["available_at"].dt.tz_convert("Europe/Berlin").tolist())


## Zulässige Preisrevisionen vor dem Merge auswählen

In [ ]:
eligible = prices[(prices["available_at"] <= cutoff)
                  & (prices["currency"] == "EUR")].copy()
price_keys = ["product_id", "price_date", "currency"]
selected_prices = eligible.sort_values(
    ["product_id", "price_date", "currency", "available_at", "revision_id"]
).drop_duplicates(subset=price_keys, keep="last")
print(selected_prices.to_string(index=False))
print(selected_prices.duplicated(subset=price_keys).sum())


## Merge: Bestandsgrain erhalten und Treffer prüfen

In [ ]:
inventory["currency"] = "EUR"
inventory["currency"] = inventory["currency"].astype("string")
inventory_keys = ["warehouse_id", "product_id", "snapshot_date"]
print(inventory.duplicated(subset=inventory_keys).sum())
print(inventory[inventory_keys].isna().sum().to_string())
print(selected_prices[price_keys].isna().sum().to_string())


In [ ]:
valued = inventory.merge(selected_prices,
    how="left",
    left_on=["product_id", "snapshot_date", "currency"],
    right_on=["product_id", "price_date", "currency"],
    validate="many_to_one", indicator=True)
valued["value_eur"] = valued["quantity"] * valued["price"]
valued["price_missing"] = valued["price"].isna()
print(valued[["warehouse_id", "product_id", "quantity", "price",
             "value_eur", "_merge", "price_missing"]].to_string(index=False))
print(len(inventory), len(valued))


## Abdeckung und Teilwert gemeinsam ausweisen

In [ ]:
priced_count = valued["price"].notna().sum()
known_part = valued["value_eur"].sum(min_count=1)
print(priced_count, len(valued), known_part)
if valued["price_missing"].sum() == 0:
    total_value = known_part
else:
    total_value = pd.NA
print(total_value)


## GroupBy: erst den Ausgabegrain festlegen

In [ ]:
warehouse_values = valued.groupby("warehouse_id", dropna=False)["value_eur"].sum(min_count=1)
warehouse_rows = valued.groupby("warehouse_id", dropna=False)["quantity"].size()
warehouse_priced = valued.groupby("warehouse_id", dropna=False)["price"].count()
warehouse_summary = pd.DataFrame({"rows":warehouse_rows,
    "priced_rows":warehouse_priced, "known_value_eur":warehouse_values})
print(warehouse_summary.to_string())


## Einen späteren Informationsstand getrennt berechnen

In [ ]:
late_cutoff = pd.Timestamp("2026-09-15T12:00:00Z")
late_eligible = prices[(prices["available_at"] <= late_cutoff)
                       & (prices["currency"] == "EUR")]
late_prices = late_eligible.sort_values(
    ["product_id", "price_date", "currency", "available_at", "revision_id"]
).drop_duplicates(subset=price_keys, keep="last")
late_valued = inventory.merge(late_prices,
    how="left", left_on=["product_id", "snapshot_date", "currency"],
    right_on=["product_id", "price_date", "currency"],
    validate="many_to_one", indicator=True)
late_valued["value_eur"] = late_valued["quantity"] * late_valued["price"]
print(late_valued[["warehouse_id", "product_id", "price", "value_eur"]].to_string(index=False))
print(late_valued["price"].notna().sum(), late_valued["value_eur"].sum(min_count=1))


## Missing erhalten, wegfiltern oder als 0 ausgeben

In [ ]:
lost_rows = valued[valued["price"].notna()]
false_zero = valued.copy()
false_zero["price"] = false_zero["price"].fillna(0)
false_zero["value_eur"] = false_zero["quantity"] * false_zero["price"]
print(len(lost_rows), len(valued))
print(false_zero[["product_id", "price", "value_eur"]].to_string(index=False))


## Duplikate und Join-Multiplikation diagnostizieren

In [ ]:
duplicate_prices = pd.concat([selected_prices, selected_prices], ignore_index=True)
duplicate_prices["key_duplicate"] = duplicate_prices.duplicated(
    subset=price_keys, keep=False)
print(duplicate_prices[["product_id", "price", "key_duplicate"]].to_string(index=False))
multiplied = inventory.merge(duplicate_prices.drop(columns="key_duplicate"),
    how="left", left_on=["product_id", "snapshot_date", "currency"],
    right_on=["product_id", "price_date", "currency"])
multiplied["value_eur"] = multiplied["quantity"] * multiplied["price"]
print(len(multiplied), multiplied["value_eur"].sum(min_count=1))


In [ ]:
# Dieser Block bricht im Buch absichtlich ab (doppelter Preis).
try:
    inventory.merge(duplicate_prices,
        how="left", left_on=["product_id", "snapshot_date", "currency"],
        right_on=["product_id", "price_date", "currency"], validate="many_to_one")
except pd.errors.MergeError as fehler:
    print(type(fehler).__name__ + ": " + str(fehler))


## Ein richtiger Typ verhindert keine falsche Skalierung

In [ ]:
scale_error = selected_prices.copy()
scale_error["price"] = scale_error["price"] * 100
print(scale_error[["product_id", "price"]].to_string(index=False))
print(10 * scale_error["price"].sum())


## Strings nur nach einer vereinbarten Regel bereinigen

In [ ]:
currency_check = pd.Series(["EUR", " eur ", pd.NA], dtype="string")
print(currency_check.str.strip().str.upper().tolist())


## Pivot: eine Vergleichsansicht ohne neue Beobachtungen

In [ ]:
quantity_matrix = inventory.pivot(
    index="warehouse_id", columns="product_id", values="quantity")
print(quantity_matrix.to_string())


## Zeitlicher Merge: die letzte bekannte Revision treffen

In [ ]:
requests = inventory.rename(columns={"snapshot_date":"price_date"}).copy()
requests["cutoff"] = pd.Series([cutoff, cutoff], dtype="datetime64[ns, UTC]")
asof_valued = pd.merge_asof(requests.sort_values("cutoff"),
    prices.sort_values("available_at"), left_on="cutoff", right_on="available_at",
    by=["product_id", "price_date", "currency"], direction="backward")
print(asof_valued[["warehouse_id", "product_id", "price"]].to_string(index=False))


## Resampling und Rolling an einer kleinen Messreihe

In [ ]:
measurements = pd.DataFrame({
    "time": pd.to_datetime(["2026-09-14T08:00:00Z", "2026-09-14T08:30:00Z",
        "2026-09-14T09:00:00Z", "2026-09-14T09:30:00Z"], utc=True),
    "value": [10, 14, 12, 16]})
measurement_series = measurements.set_index("time")["value"]
hourly = measurement_series.resample("1h", closed="left", label="left").mean()
rolling = measurement_series.rolling(window=2, min_periods=2).mean()
print(hourly.to_string())
print(rolling.to_string())


## Exploration: tatsächliche Werte und Auffälligkeiten beschreiben

In [ ]:
orders["received_at"] = pd.to_datetime(orders["received_at"], utc=True)
orders["processed_at"] = pd.to_datetime(orders["processed_at"], utc=True)
orders["duration_min"] = (orders["processed_at"] - orders["received_at"]).dt.total_seconds() / 60
print(orders[["order_id", "processing_group", "duration_min"]].to_string(index=False))
print(orders["duration_min"].describe().to_string())
print(orders["processing_group"].value_counts(dropna=False).to_string())
print(valued.isna().sum().to_string())


## Effizienz: Speicher und Verarbeitung passend begrenzen

In [ ]:
print(inventory.memory_usage(index=True, deep=True).sum())


## Diagramme aus Abbildung 8.1 und 8.2

So entstehen die beiden Abbildungen des Kapitels.

In [ ]:
import matplotlib.pyplot as plt
axis = orders["duration_min"].plot.hist(bins=[0, 60, 120, 180],
    color="gray", edgecolor="black")
axis.set_xlabel("Bearbeitungsdauer (Minuten)")
axis.set_ylabel("Anzahl Bestellungen")
axis.set_title("Vier Referenzbestellungen")
axis.set_yticks([0, 1, 2])
plt.tight_layout()
plt.show()
plt.close()
axis = measurement_series.plot(marker="o", color="black")
axis.set_xlabel("Messzeit (UTC)")
axis.set_ylabel("Messwert (neutrale Einheit)")
axis.set_title("Vier beobachtete Messwerte")
plt.tight_layout()
plt.show()
plt.close()
